In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [4]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [7]:
label_encoder_gen=LabelEncoder()
data['Gender']=label_encoder_gen.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [11]:
from sklearn.preprocessing import OneHotEncoder
onehot_geo=OneHotEncoder()
geo_encoder=onehot_geo.fit_transform(data[['Geography']]).toarray()

In [12]:
geo_encoder

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [15]:
onehot_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [19]:
geo_encoded_df=pd.DataFrame(geo_encoder, columns=onehot_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [21]:
##combining one hot encoder columns with original data
data=pd.concat([data.drop('Geography',axis=1), geo_encoded_df],axis=1)
data.head(2)

KeyError: "['Geography'] not found in axis"

In [22]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [24]:
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gen, file)

with open('label_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_geo, file)


In [26]:
#dividing dataset into independent and dependent features
x=data.drop('Exited',axis=1)
y=data['Exited']

x_train, x_test, y_train, y_test=train_test_split(x,y, test_size=0.2, random_state=42)

#scaling
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.fit_transform(x_test)

In [27]:
x_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [28]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler, file)

In [29]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [30]:
!git init

Initialized empty Git repository in C:/Users/Ritesh Kandwal/Documents/4 STUDY/PYTHON/projects/udemyproject/.git/


In [31]:
!git remote add origin https://github.com/ritesh263/New-Project

In [32]:
!git add .

In [33]:
!git commit -m "Added notebook and updates"

[master (root-commit) 9928c22] Added notebook and updates
 7 files changed, 13029 insertions(+)
 create mode 100644 .ipynb_checkpoints/experiments-checkpoint.ipynb
 create mode 100644 Churn_Modelling.csv
 create mode 100644 experiments.ipynb
 create mode 100644 label_encoder_gender.pkl
 create mode 100644 label_encoder_geo.pkl
 create mode 100644 requirements.txt
 create mode 100644 scaler.pkl


In [37]:
!git branch -M main
!git push -u origin main

To https://github.com/ritesh263/New-Project
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/ritesh263/New-Project'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [38]:
!git pull origin main --rebase
!git push origin main

error: cannot pull with rebase: You have unstaged changes.
error: Please commit or stash them.
To https://github.com/ritesh263/New-Project
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/ritesh263/New-Project'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [39]:
!git checkout main
!git merge master
!git push origin main

M	experiments.ipynb


Already on 'main'
merge: master - not something we can merge

Did you mean this?
	origin/master
To https://github.com/ritesh263/New-Project
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/ritesh263/New-Project'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [40]:
!git push origin main

To https://github.com/ritesh263/New-Project
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/ritesh263/New-Project'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [41]:
!git add .
!git commit -m "update notebook"

[main 893abb0] update notebook
 1 file changed, 180 insertions(+), 13 deletions(-)


In [42]:
!git pull origin main --rebase

From https://github.com/ritesh263/New-Project
 * branch            main       -> FETCH_HEAD
 * [new branch]      main       -> origin/main
Rebasing (1/2)
Rebasing (2/2)
Successfully rebased and updated refs/heads/main.


In [43]:
!git push origin main

To https://github.com/ritesh263/New-Project
   efb2519..a637253  main -> main
